# 04. Batch Scoring e Célula Especializada
Simulando a execução diária em produção para encontrar os clientes com maior propensão a acionar o BACEN.

In [ ]:
# Como não estamos em um Cluster Databricks ML (com libs pré-instaladas),
# precisamos instalar as dependências na máquina em tempo de execução:
%pip install xgboost scikit-learn mlflow


In [ ]:
import mlflow
from pyspark.sql.functions import col

# Carregando modelo do MLflow Registry
print("Puxando o último modelo de produção...")
try:
    model_uri = "models:/workspace.bacen_mlops.BacenRiskXGBoost/latest"
    loaded_model = mlflow.pyfunc.load_model(model_uri)
    HAS_MLFLOW = True
except Exception as e:
    print(f"Aviso: Não encontrou modelo registrado. Erro: {e}")
    HAS_MLFLOW = False

In [ ]:

if HAS_MLFLOW:
    print("Aplicando o modelo nos dados da D-0...")
    # Na vida real, carregaríamos novos clientes sem a flag target_bacen
    # Vamos simular carregando o próprio parquet
    BASE_PATH = "/Volumes/workspace/default/raw_data"
    df_hoje = spark.table("workspace.bacen_mlops.call_center_features")
    
    # MLflow aceita Spark UDF para rodar distribuído!
    predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri)
    
    df_scored = df_hoje.withColumn(
        "bacen_risk_score", 
        predict_udf(*[col(c) for c in df_hoje.columns if c not in ("customer_id", "target_bacen")])
    )
    
    print("ROTEAMENTO PARA A CÉLULA ESPECIALIZADA")
    print("Pegando o TOP 10 clientes com maior risco (Score = 1) para a equipe ligar preventivamente HOJE:")
    
    df_scored.filter(col("bacen_risk_score") == 1) \
             .select("customer_id", "max_calls_last_7d", "unresolved_issue_recently", "bacen_risk_score") \
             .orderBy(col("max_calls_last_7d").desc()) \
             .show(10)
else:
    print("Execute o notebook de treinamento primeiro para gerar o modelo.")
